In [0]:
%sql

CREATE OR REPLACE TEMP VIEW parsed_llm_results AS

SELECT
    patient_id,
    specialty,

    generated_summary.result AS raw_result,

    generated_summary.errorMessage AS error_message,

    generated_at

FROM hospital_poc.silver.llm_summary_results;

In [0]:
%sql

SELECT *
FROM parsed_llm_results
ORDER BY patient_id
LIMIT 10;

patient_id,specialty,raw_result,error_message,generated_at
1,oncology,"{""medications"": ""Not documented"", ""diagnoses"": ""Metastatic Adenocarcinoma of unknown primary (ICD: C80.0)"", ""hospital_course"": ""Underwent mastectomy followed by chemotherapy."", ""investigations"": ""CBC shows anemia. PET scan shows metastasis."", ""vitals_and_exam"": ""Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -."", ""next_steps"": ""Regular follow-up in the oncology OPD."", ""discharge_condition"": ""Stable, afebrile."", ""summary"": ""68‑year‑old female with 8‑month history of unexplained weight loss and fatigue, gradual loss of appetite, and a 2x2 cm hard lump in the left breast. CBC revealed anemia; PET scan showed metastasis. Diagnosis: metastatic adenocarcinoma of unknown primary. Hospital course included mastectomy and chemotherapy. Discharge status: stable, afebrile. Plan: regular oncology follow‑up.""}",null,2026-09-25T08:32:56.991Z
2,cardiology,,null,2026-09-25T08:32:56.991Z
3,general_medicine,"{""medications"": ""IV antibiotics"", ""diagnoses"": ""Community-Acquired Pneumonia (ICD: J18.9); Gastritis"", ""hospital_course"": ""IV antibiotics administered; fever subsided"", ""investigations"": ""Endoscopy revealed gastritis"", ""vitals_and_exam"": ""Temperature 102°F; chest crepitations; pallor present"", ""next_steps"": ""Diet: bland; follow-up in 1 week"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""An 85‑year‑old female presented with 22‑day abdominal pain and productive cough. Physical exam showed fever (102°F) and chest crepitations with pallor. Endoscopy identified gastritis. She was diagnosed with community‑acquired pneumonia (ICD: J18.9). During hospitalization she received IV antibiotics, after which fever resolved. At discharge she was stable and afebrile. Discharge instructions included a bland diet and a follow‑up appointment in one week.""}",null,2026-09-25T08:32:56.991Z
4,pediatrics,"{""medications"": ""ORS, IV fluids, Zinc"", ""diagnoses"": ""Viral Exanthem (ICD: B09)"", ""hospital_course"": ""Administered ORS, IV fluids, and Zinc during hospital stay."", ""investigations"": ""CRP: Positive"", ""vitals_and_exam"": ""Dehydrated, abdomen soft; Pallor +, Icterus -, Cyanosis -, Clubbing -, Koilonychia -, Lymphadenopathy -, Edema -"", ""next_steps"": ""Diet: Breastfeed; Follow-up: 5 days"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""Patient presented with 17 days of loose motions and vomiting, high fever, and measles-like rash. Physical exam showed dehydration, soft abdomen, pallor, and no icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, or edema. CRP was positive. Diagnosed with viral exanthem (ICD B09). Treated with ORS, IV fluids, and zinc. Discharged stable and afebrile, advised breastfeeding and follow-up in 5 days.""}",null,2026-09-25T08:32:56.991Z
5,pediatrics,"{""medications"": ""Not documented"", ""diagnoses"": ""Viral Exanthem (ICD: B09)"", ""hospital_course"": ""During the hospital stay, symptomatic treatment was provided and the patient resolved."", ""investigations"": ""CRP: Positive"", ""vitals_and_exam"": ""Temperature 102°F, congested throat; pallor present, icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, and edema absent."", ""next_steps"": ""Diet: Breastfeed; follow-up in 5 days."", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""Newborn male presented with 27 days of fever and rash, mild dehydration, and recent URI. Physical exam showed fever, throat congestion, pallor, and no other abnormalities. CRP was positive. Diagnosis: viral exanthem (ICD B09). Symptomatic treatment was administered during hospitalization, leading to resolution. At discharge, the infant was stable and afebrile. The plan includes continued breastfeeding and a follow-up visit in 5 days.""}",null,2026-09-25T08:32:56.991Z
6,oncology,"{""medications"": ""palliative chemotherapy"", ""diagnoses"": ""Metastatic Adenocarcino

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW validated_llm_results AS

SELECT
    patient_id,
    specialty,
    raw_result,
    error_message,
    generated_at,

    CASE
        WHEN error_message IS NOT NULL
            THEN 'ERROR'

        WHEN raw_result IS NULL
            OR TRIM(raw_result) = ''
            THEN 'EMPTY_RESPONSE'

        ELSE 'SUCCESS'
    END AS processing_status

FROM parsed_llm_results;

In [0]:
%sql

SELECT
    processing_status,
    COUNT(*) AS record_count
FROM validated_llm_results
GROUP BY processing_status
ORDER BY processing_status;

processing_status,record_count
EMPTY_RESPONSE,64
SUCCESS,936


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW parsed_gold_data AS

SELECT
    patient_id,
    specialty,

    from_json(
        raw_result,
        'STRUCT<
            medications:STRING,
            diagnoses:STRING,
            hospital_course:STRING,
            investigations:STRING,
            vitals_and_exam:STRING,
            next_steps:STRING,
            discharge_condition:STRING,
            summary:STRING
        >'
    ) AS summary_data,

    processing_status,
    error_message,
    generated_at

FROM validated_llm_results;

In [0]:
%sql

SELECT
    patient_id,
    specialty,
    summary_data.diagnoses,
    summary_data.medications,
    summary_data.investigations,
    summary_data.hospital_course,
    summary_data.summary,
    processing_status
FROM parsed_gold_data
ORDER BY patient_id
LIMIT 10;

patient_id,specialty,diagnoses,medications,investigations,hospital_course,summary,processing_status
1,oncology,Metastatic Adenocarcinoma of unknown primary (ICD: C80.0),Not documented,CBC shows anemia. PET scan shows metastasis.,Underwent mastectomy followed by chemotherapy.,"68‑year‑old female with 8‑month history of unexplained weight loss and fatigue, gradual loss of appetite, and a 2x2 cm hard lump in the left breast. CBC revealed anemia; PET scan showed metastasis. Diagnosis: metastatic adenocarcinoma of unknown primary. Hospital course included mastectomy and chemotherapy. Discharge status: stable, afebrile. Plan: regular oncology follow‑up.",SUCCESS
2,cardiology,null,null,null,null,null,EMPTY_RESPONSE
3,general_medicine,Community-Acquired Pneumonia (ICD: J18.9); Gastritis,IV antibiotics,Endoscopy revealed gastritis,IV antibiotics administered; fever subsided,"An 85‑year‑old female presented with 22‑day abdominal pain and productive cough. Physical exam showed fever (102°F) and chest crepitations with pallor. Endoscopy identified gastritis. She was diagnosed with community‑acquired pneumonia (ICD: J18.9). During hospitalization she received IV antibiotics, after which fever resolved. At discharge she was stable and afebrile. Discharge instructions included a bland diet and a follow‑up appointment in one week.",SUCCESS
4,pediatrics,Viral Exanthem (ICD: B09),"ORS, IV fluids, Zinc",CRP: Positive,"Administered ORS, IV fluids, and Zinc during hospital stay.","Patient presented with 17 days of loose motions and vomiting, high fever, and measles-like rash. Physical exam showed dehydration, soft abdomen, pallor, and no icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, or edema. CRP was positive. Diagnosed with viral exanthem (ICD B09). Treated with ORS, IV fluids, and zinc. Discharged stable and afebrile, advised breastfeeding and follow-up in 5 days.",SUCCESS
5,pediatrics,Viral Exanthem (ICD: B09),Not documented,CRP: Positive,"During the hospital stay, symptomatic treatment was provided and the patient resolved.","Newborn male presented with 27 days of fever and rash, mild dehydration, and recent URI. Physical exam showed fever, throat congestion, pallor, and no other abnormalities. CRP was positive. Diagnosis: viral exanthem (ICD B09). Symptomatic treatment was administered during hospitalization, leading to resolution. At discharge, the infant was stable and afebrile. The plan includes continued breastfeeding and a follow-up visit in 5 days.",SUCCESS
6,oncology,null,null,null,null,null,SUCCESS
7,oncology,Carcinoma of the Breast (ICD: C50.9),Not documented,CBC shows anemia; PET scan shows metastasis,Underwent mastectomy followed by chemotherapy,"A 24‑year‑old female presented with unexplained weight loss, fatigue, and a painless 2x2 cm hard lump in the left breast. Examination revealed pallor and a hard mass; no lymphadenopathy or edema. CBC indicated anemia, and PET imaging demonstrated metastasis. She was diagnosed with carcinoma of the breast (ICD C50.9). During hospitalization she underwent mastectomy and received chemotherapy. At discharge she was stable and afebrile, with a plan for regular oncology follow‑up.",SUCCESS
8,neurology,null,null,null,null,null,EMPTY_RESPONSE
9,general_medicine,null,null,null,null,null,SUCCESS
10,cardiology,NSTEMI (ICD: I21.4),"Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS",ECG: ST elevation; Trop-I: Positive; Echo: EF 51%; Angiogram: 79% blockage in LAD,"Antiarrhythmics given, rhythm stabilized","An 8‑month‑old male presented with palpitations and dizziness for 26 days. Physical exam revealed elevated BP 175/100 mmHg, normal heart sounds, no murmur. ECG showed ST elevation, troponin I positive, echo EF 51%, angiogram 79% LAD blockage. Diagnosed with NSTEMI (ICD I21.4). During hospitalization, antiarrhythmics were administered and rhythm stabilized. Discharged stable and afebrile. Medications at discharge: Aspirin 75 mg daily and Atorvastatin 40 mg nightly.",SUCCESS


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW final_validated_results AS

SELECT
    patient_id,
    specialty,

    summary_data.diagnoses,
    summary_data.medications,
    summary_data.investigations,
    summary_data.vitals_and_exam,
    summary_data.hospital_course,
    summary_data.discharge_condition,
    summary_data.next_steps,
    summary_data.summary,

    processing_status AS initial_status,
    error_message,
    generated_at,

    CASE
        WHEN error_message IS NOT NULL
            THEN 'ERROR'

        WHEN processing_status = 'EMPTY_RESPONSE'
            THEN 'EMPTY_RESPONSE'

        WHEN summary_data IS NULL
            THEN 'MALFORMED_RESPONSE'

        WHEN summary_data.summary IS NULL
            OR TRIM(summary_data.summary) = ''
            THEN 'MALFORMED_RESPONSE'

        ELSE 'SUCCESS'
    END AS final_status

FROM parsed_gold_data;

In [0]:
%sql

SELECT
    final_status,
    COUNT(*) AS record_count
FROM final_validated_results
GROUP BY final_status
ORDER BY final_status;

final_status,record_count
EMPTY_RESPONSE,64
MALFORMED_RESPONSE,157
SUCCESS,779


In [0]:
%sql

SELECT
    final_status,
    COUNT(*) AS record_count
FROM final_validated_results
WHERE final_status != 'SUCCESS'
GROUP BY final_status;

final_status,record_count
EMPTY_RESPONSE,64
MALFORMED_RESPONSE,157


In [0]:
%sql

SELECT
    patient_id,
    specialty,
    generated_summary.result AS raw_result,
    generated_summary.errorMessage AS error_message,
    generated_at
FROM hospital_poc.silver.llm_summary_results
WHERE generated_summary.result IS NULL
   OR TRIM(generated_summary.result) = ''
   OR patient_id IN (
       SELECT patient_id
       FROM final_validated_results
       WHERE final_status = 'MALFORMED_RESPONSE'
   )
ORDER BY patient_id
LIMIT 20;

patient_id,specialty,raw_result,error_message,generated_at
2,cardiology,,null,2026-09-25T08:32:56.991Z
6,oncology,"{""medications"": ""palliative chemotherapy"", ""diagnoses"": ""Metastatic Adenocarcinoma of unknown primary (ICD: C80.0)"", ""hospital_course"": ""Patient started on palliative chemotherapy; condition stable during stay."", ""investigations"": ""Mammogram and biopsy confirmed malignancy."", ""vitals_and_exam"": ""Physical exam: 2x2 cm hard lump in upper outer quadrant of left breast; pallor present; icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, edema absent."", ""next_steps"": ""Advised high protein diet."", ""discharge_condition"": ""Stable, afebrile."", ""summary"": ""24-year-old female with 23-week history of painless hard lump in left breast. Physical exam revealed 2x2 cm hard mass in upper outer quadrant; pallor present; no icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, or edema. Mammogram and biopsy confirmed malignancy; diagnosis",null,2026-09-25T08:32:56.991Z
8,neurology,,null,2026-09-25T08:32:56.991Z
9,general_medicine,"{""medications"": ""Tab. Amoxicillin 500mg TDS; IV antibiotics"", ""diagnoses"": ""Community-Acquired Pneumonia (ICD: J18.9)"", ""hospital_course"": ""IV antibiotics administered; fever subsided; patient felt better; discharged stable and afebrile; advised to complete course of Amoxicillin 500mg TDS"", ""investigations"": ""CBC with high WBC; Chest X-ray showing pneumonia"", ""vitals_and_exam"": ""Temperature 102°F; chest crepitations; pallor present; icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, edema absent"", ""next_steps"": ""Complete full course of prescribed antibiotics"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""1",null,2026-09-25T08:32:56.991Z
11,neurology,"{""medications"": ""Tab. Clopidogrel 75mg OD."", ""diagnoses"": ""Migraine (ICD: G43.9)"", ""hospital_course"": ""IV thrombolysis, physiotherapy. Improved."", ""investigations"": ""MRI: Left MCA infarct; CT: No bleed."", ""vitals_and_exam"": ""Power: 3/5 right limbs; Reflexes: Hyperactive. Pallor: +, Icter",null,2026-09-25T08:32:56.991Z
13,cardiology,,null,2026-09-25T08:32:56.991Z
15,neurology,"{""medications"": ""Tab. Clopidogrel 75mg OD."", ""diagnoses"": ""Migraine (ICD: G43.9)"", ""hospital_course"": ""During the hospital stay, IV thrombolysis, physiotherapy. Improved."", ""investigations"": ""MRI: Left MCA infarct; CT: No bleed."", ""vitals_and_exam"": ""Cranial nerves: Intact. Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -."", ""next_steps"": ""Take Clopidogrel 75mg OD on time."", ""discharge_condition"": ""Stable, afebrile."", ""summary"": ""Baby Vihaan, 35-year-old male, presented with right-sided weakness and speech difficulty for 10 days, sudden onset with facial droop and breathing difficulty. Physical exam showed intact cranial nerves; pallor present, other signs absent. MRI revealed left MCA infarct; CT negative for bleed. Diagnosis recorded as migraine (ICD-10 G43.9). During hospitalization, received IV thrombolysis and physiotherapy, with improvement. Discharged stable and afebrile, advised to take Clopidogrel 75mg daily on time.""",null,2026-09-25T08:32:56.991Z
18,neurology,"{""medications"": ""Tab. Clopidogrel 75mg OD."", ""diagnoses"": ""Migraine (ICD: G43.9)"", ""hospital_course"": ""IV thrombolysis and physiotherapy administered; patient improved."", ""investigations"": ""EEG revealed abnormal spikes."", ""vitals_and_exam"": ""Power 3/5 in right limbs; hyperactive reflexes. Pallor present; icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, edema absent."", ""next_steps"": ""Continue Tab. Clopidogrel 75mg OD."", ""discharge_condition"": ""Stable and afebrile."", ""summary"": ""A 3‑year‑old male with a 29‑day history of severe headache, nausea, photophobia, and migraine‑like features presented. Past history of childhood epilepsy. Examination revealed right‑sided weakness (3/5 power)",null,2026-09-2

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW retry_candidates AS

SELECT
    s.patient_id,
    s.specialty,
    s.transcript
FROM hospital_poc.silver.patient_transcripts s
INNER JOIN final_validated_results v
    ON s.patient_id = v.patient_id
WHERE v.final_status IN (
    'EMPTY_RESPONSE',
    'MALFORMED_RESPONSE'
);

SELECT
    COUNT(*) AS retry_count
FROM retry_candidates;

retry_count
221


In [0]:
%sql

CREATE OR REPLACE TABLE hospital_poc.silver.llm_retry_results AS

SELECT
    patient_id,
    specialty,

    ai_query(
        'system.ai.gpt-oss-20b',

        CONCAT(
            'Create a factual clinical summary from this transcript.

Return JSON with exactly these fields:
diagnoses, medications, investigations, vitals_and_exam,
hospital_course, discharge_condition, next_steps, summary.

Rules:
- Use ONLY facts explicitly present in the transcript.
- Never invent or infer medical information.
- If a field is absent, write "Not documented".
- The summary must be less than 250 words.
- Keep every field concise.
- Complete the JSON response.

Transcript:
',
            transcript
        ),

        modelParameters => named_struct(
            'max_tokens', 1000,
            'temperature', 0.0
        ),

        responseFormat => '{
          "type": "json_schema",
          "json_schema": {
            "name": "clinical_summary_retry",
            "schema": {
              "type": "object",
              "properties": {
                "diagnoses": {"type": "string"},
                "medications": {"type": "string"},
                "investigations": {"type": "string"},
                "vitals_and_exam": {"type": "string"},
                "hospital_course": {"type": "string"},
                "discharge_condition": {"type": "string"},
                "next_steps": {"type": "string"},
                "summary": {"type": "string"}
              },
              "required": [
                "diagnoses",
                "medications",
                "investigations",
                "vitals_and_exam",
                "hospital_course",
                "discharge_condition",
                "next_steps",
                "summary"
              ],
              "additionalProperties": false
            },
            "strict": true
          }
        }',

        failOnError => false

    ) AS retry_result,

    current_timestamp() AS retry_generated_at

FROM retry_candidates;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS retry_records
FROM hospital_poc.silver.llm_retry_results;

retry_records
221


In [0]:
%sql

DESCRIBE hospital_poc.silver.llm_retry_results;

col_name,data_type,comment
patient_id,bigint,null
specialty,string,null
retry_result,struct,null
retry_generated_at,timestamp,null


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW parsed_retry_results AS

SELECT
    patient_id,
    specialty,

    retry_result.result AS raw_result,
    retry_result.errorMessage AS error_message,

    retry_generated_at

FROM hospital_poc.silver.llm_retry_results;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW validated_retry_results AS

SELECT
    patient_id,
    specialty,

    from_json(
        raw_result,
        'STRUCT<
            medications:STRING,
            diagnoses:STRING,
            hospital_course:STRING,
            investigations:STRING,
            vitals_and_exam:STRING,
            next_steps:STRING,
            discharge_condition:STRING,
            summary:STRING
        >'
    ) AS summary_data,

    error_message,
    retry_generated_at

FROM parsed_retry_results;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW final_retry_results AS

SELECT
    patient_id,
    specialty,
    summary_data,
    error_message,
    retry_generated_at,

    CASE
        WHEN error_message IS NOT NULL
            THEN 'RETRY_ERROR'

        WHEN summary_data IS NULL
            THEN 'RETRY_EMPTY_OR_MALFORMED'

        WHEN summary_data.summary IS NULL
            OR TRIM(summary_data.summary) = ''
            THEN 'RETRY_MALFORMED'

        ELSE 'RETRY_SUCCESS'
    END AS retry_status

FROM validated_retry_results;

In [0]:
%sql

SELECT
    retry_status,
    COUNT(*) AS record_count
FROM final_retry_results
GROUP BY retry_status
ORDER BY retry_status;

retry_status,record_count
RETRY_EMPTY_OR_MALFORMED,23
RETRY_MALFORMED,56
RETRY_SUCCESS,142


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW final_successful_results AS

-- Original successful results
SELECT
    patient_id,
    specialty,
    diagnoses,
    medications,
    investigations,
    vitals_and_exam,
    hospital_course,
    discharge_condition,
    next_steps,
    summary,
    generated_at,
    'ORIGINAL_SUCCESS' AS processing_status
FROM final_validated_results
WHERE final_status = 'SUCCESS'

UNION ALL

-- Successful retry results
SELECT
    patient_id,
    specialty,
    summary_data.diagnoses,
    summary_data.medications,
    summary_data.investigations,
    summary_data.vitals_and_exam,
    summary_data.hospital_course,
    summary_data.discharge_condition,
    summary_data.next_steps,
    summary_data.summary,
    retry_generated_at AS generated_at,
    'RETRY_SUCCESS' AS processing_status
FROM final_retry_results
WHERE retry_status = 'RETRY_SUCCESS';

In [0]:
%sql
SELECT
    processing_status,
    COUNT(*) AS record_count
FROM final_successful_results
GROUP BY processing_status;

processing_status,record_count
ORIGINAL_SUCCESS,779
RETRY_SUCCESS,142


In [0]:
%sql
SELECT COUNT(*) AS total_successful
FROM final_successful_results;

total_successful
921


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_poc.gold.patient_clinical_summaries AS

SELECT
    s.patient_id,
    s.specialty,
    s.transcript AS original_transcript,

    f.diagnoses,
    f.medications,
    f.investigations,
    f.vitals_and_exam,
    f.hospital_course,
    f.discharge_condition,
    f.next_steps,
    f.summary AS clinical_summary,

    f.processing_status,
    f.generated_at,
    current_timestamp() AS gold_created_at

FROM hospital_poc.silver.patient_transcripts s

INNER JOIN final_successful_results f
    ON s.patient_id = f.patient_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*)
FROM hospital_poc.gold.patient_clinical_summaries;

COUNT(*)
921


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_poc.gold.llm_failed_records AS

SELECT
    s.patient_id,
    s.specialty,
    s.transcript AS original_transcript,

    p.raw_result AS raw_llm_response,

    r.retry_status AS failure_status,
    p.error_message,
    p.retry_generated_at AS processed_at

FROM hospital_poc.silver.patient_transcripts s

INNER JOIN parsed_retry_results p
    ON s.patient_id = p.patient_id

INNER JOIN final_retry_results r
    ON s.patient_id = r.patient_id

WHERE r.retry_status IN (
    'RETRY_EMPTY_OR_MALFORMED',
    'RETRY_MALFORMED'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    failure_status,
    COUNT(*) AS record_count
FROM hospital_poc.gold.llm_failed_records
GROUP BY failure_status;

failure_status,record_count
RETRY_MALFORMED,56
RETRY_EMPTY_OR_MALFORMED,23


In [0]:
%sql
SELECT *
FROM hospital_poc.gold.llm_failed_records
LIMIT 5;

patient_id,specialty,original_transcript,raw_llm_response,failure_status,error_message,processed_at
8,neurology,"Patient ID: 8. Specialty: neurology. Patient name: Dinesh Kulkarni. Age/Sex: Newborn/M. The patient presented with Right side weakness aur bolne mein difficulty for 23 days.. The history of present illness was Migraine-like, photophobia. saans lene mein takleef thi.. Relevant past history included Hypertension, smoker.. Physical examination showed Power: 3/5 right limbs; Reflexes: Hyperactive. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed MRI: Left MCA infarct; CT: No bleed.. The diagnosis was Ischemic Stroke (ICD: I63.9).. During the hospital stay, IV thrombolysis, physiotherapy. Improved.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Avoid triggers; Follow-up: 10 days. - pura course complete karna..","{""medications"": ""Not documented"", ""diagnoses"": ""Ischemic Stroke (ICD: I63.9)"", ""hospital_course"": ""IV thrombolysis and physiotherapy; patient improved"", ""investigations"": ""MRI: Left MCA infarct; CT: No bleed"", ""vitals_and_exam"": ""Power 3/5 right limbs",RETRY_MALFORMED,null,2026-09-28T06:28:12.603Z
15,neurology,"Patient ID: 15. Specialty: neurology. Patient name: Baby Vihaan. Age/Sex: 35/M. The patient presented with Right side weakness aur bolne mein difficulty for 10 days.. The history of present illness was Sudden onset, with facial droop. saans lene mein takleef thi.. Relevant past history included Epilepsy since childhood.. Physical examination showed Cranial nerves: Intact. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed MRI: Left MCA infarct; CT: No bleed.. The diagnosis was Migraine (ICD: G43.9).. During the hospital stay, IV thrombolysis, physiotherapy. Improved.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Meds: Tab. Clopidogrel 75mg OD. - Dawa time pe lena..","{""medications"": ""Clopidogrel 75mg OD"", ""diagnoses"": ""Migraine (ICD: G43.9)"", ""hospital_course"": ""IV thrombolysis and physiotherapy; patient improved"", ""investigations"": ""MRI: Left MCA infarct; CT: No bleed"", ""vitals_and_exam"": ""Cranial nerves intact; Pallor +; Icterus -; Cyanosis -; Clubbing -; Koil",RETRY_MALFORMED,null,2026-09-28T06:28:12.603Z
20,neurology,"Patient ID: 20. Specialty: neurology. Patient name: Aman Verma. Age/Sex: Newborn/M. The patient presented with Right side weakness aur bolne mein difficulty for 2 days.. The history of present illness was Migraine-like, photophobia. Patient ko fever tha.. Relevant past history included Epilepsy since childhood.. Physical examination showed Cranial nerves: Intact. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed EEG: Abnormal spikes.. The diagnosis was Ischemic Stroke (ICD: I63.9).. During the hospital stay, IV thrombolysis, physiotherapy. Improved.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Meds: Tab. Clopidogrel 75mg OD..","{""medications"": ""Clopidogrel 75mg OD"", ""diagnoses"": ""Ischemic Stroke (ICD: I63.9)"", ""hospital_course"": ""IV thrombolysis and physiotherapy; patient improved"", ""investigations"": ""EEG: Abnormal spikes"", ""vitals_and_exam"": ""Cranial nerves intact; pallor +; icterus -; cyanosis -; clubbing -; koilonychia -; lymphadenopathy -; edema -"", ""next_steps"": ""Not documented"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""Newborn male Aman Verma presented with right-sided weakness and speech difficulty for 2",RETRY_MALFORMED,null,2026-09-28T06:28:12.603Z
21,general_medicine,"Patient ID: 21. Specialty: general_medicine. Patient name: Priya Singh. Age/Sex: Newborn/M. The patient presented with Abdominal pain for 26 days.. The history of present il